# Qwen3-ASR 0.6B LoRA Levantine Custom Run: Palestine + Jordan + North Levantine
This notebook is a near plug-and-play copy of the Whisper Medium Levantine custom run, with only the Qwen3-ASR 0.6B changes needed for processor/model loading, instruction-style labels, and safer generation decoding.


## 1. Config

In [2]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path("/home/MohammadNabulsi/whisper")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import (
    NOTEBOOK_PATH,
    RUN_DIR,
    SMOKE_NOTEBOOK_PATH,
    config_snapshot,
    ensure_run_layout,
    make_config,
    setup_logging,
)

RUN_SMOKE_TEST = False
RUN_BASELINE_BEFORE_TRAIN = True
RUN_POST_TRAIN_EVAL = True

config = make_config(
    smoke_mode=RUN_SMOKE_TEST,
    num_train_epochs=50,
    run_baseline_before_train=RUN_BASELINE_BEFORE_TRAIN,
    run_post_train_eval=RUN_POST_TRAIN_EVAL,
)
ensure_run_layout()
log_path = setup_logging()
print(json.dumps(config_snapshot(config), ensure_ascii=False, indent=2))
print("Notebook path:", SMOKE_NOTEBOOK_PATH if RUN_SMOKE_TEST else NOTEBOOK_PATH)
print("Log path:", log_path)


{
  "model_name": "Qwen/Qwen3-ASR-0.6B",
  "run_id": "qwen3_asr_0_6b_levantine_custom_streaming_5minckpt",
  "sample_rate": 16000,
  "min_audio_seconds": 0.3,
  "drop_audio_at_or_above_seconds": null,
  "train_batch_size": 4,
  "eval_batch_size": 16,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0001,
  "warmup_ratio": 0.05,
  "weight_decay": 0.01,
  "max_grad_norm": 1.0,
  "num_train_epochs": 50,
  "logging_steps": 0,
  "checkpoint_every_minutes": 5,
  "save_total_limit": 6,
  "generation_max_new_tokens": 256,
  "split_seed": 42,
  "train_dataloader_num_workers": 4,
  "language": "Arabic",
  "smoke_mode": false,
  "run_baseline_before_train": true,
  "run_post_train_eval": true,
  "force_rebuild_manifests": false,
  "resume_from_checkpoint": false,
  "use_bf16": true,
  "use_fp16": false,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "qwen_system_prompt": "Transcribe this audio into Arabic.",
  "run_dir": "/home/MohammadNabulsi/whisper/Runs/qwen3_asr_0_6b_le

## 2. Build Filtered Mini Manifests

In [3]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import prepare_manifests

manifest_state = prepare_manifests(config)
selection_summary = manifest_state["selection_summary"]
print(json.dumps(selection_summary, ensure_ascii=False, indent=2))


2026-06-23 18:08:44,340 | INFO | qwen3_asr_levantine_run | prepared split=train rows=1860 kept=1859 dropped_empty=0 dropped_duration=1
2026-06-23 18:08:44,373 | INFO | qwen3_asr_levantine_run | prepared split=eval_pool rows=1515 kept=1514 dropped_empty=0 dropped_duration=1
2026-06-23 18:08:44,416 | INFO | qwen3_asr_levantine_run | prepared split=eval_pool rows=172 kept=172 dropped_empty=0 dropped_duration=0


{
  "run_id": "qwen3_asr_0_6b_levantine_custom_streaming_5minckpt",
  "drop_segments_at_or_above_seconds": null,
  "full_counts_after_filter": {
    "train": 1859,
    "eval_parquet_pool": 1514,
    "eval_arrow_pool": 172
  },
  "selected_counts": {
    "train": 1859,
    "val": 843,
    "test": 843
  },
  "selected_hours": {
    "train": 8.697106354091638,
    "val": 2.704268942114574,
    "test": 2.572148800653972
  },
  "train_by_source_group": {
    "casablanca_levantine_train": {
      "rows": 1514,
      "hours": 2.0004557985360836
    },
    "omnilingual_apc_north_levantine_train": {
      "rows": 345,
      "hours": 6.696650555555555
    }
  },
  "val_by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      "hours": 0.9691543645682779
    },
    "omnilingual_apc_north_levantine_eval_pool": {
      "rows": 86,
      "hours": 1.7351145775462962
    }
  },
  "test_by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      "hour

## 3. Baseline Qwen3-ASR 0.6B Predictions


In [3]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import load_qwen_bundle, run_predictions

baseline_bundle = load_qwen_bundle(config)
baseline_test_metrics = None
if config.run_baseline_before_train:
    baseline_test_metrics = run_predictions(
        manifest_state["test_rows"],
        config,
        baseline_bundle,
        name="baseline_test_predictions",
    )
print(json.dumps(baseline_test_metrics, ensure_ascii=False, indent=2))


2026-06-23 17:06:01,125 | INFO | qwen3_asr_levantine_run | Loading Qwen3-ASR from /home/MohammadNabulsi/.cache/huggingface/hub/models--Qwen--Qwen3-ASR-0.6B/snapshots/5eb144179a02acc5e5ba31e748d22b0cf3e303b0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-06-23 17:06:04,867 | INFO | qwen3_asr_levantine_run | model=Qwen3ASRForConditionalGeneration processor=Qwen3ASRProcessor adapter=None
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `

{
  "num_predictions": 843,
  "prediction_path": "/home/MohammadNabulsi/whisper/Runs/qwen3_asr_0_6b_levantine_custom_streaming_5minckpt/eval_predictions/baseline_test_predictions.jsonl",
  "wer": 0.6402794778420853,
  "cer": 0.2616894349262296,
  "wer_loose": 0.6126387234307413,
  "cer_loose": 0.2499682348954728,
  "wer_no_punct": 0.6156147951234372,
  "cer_no_punct": 0.23742627653273468,
  "total_hours": 2.572148800653972,
  "by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      "wer": 0.6403470239694935,
      "cer": 0.2580460354944333
    },
    "omnilingual_apc_north_levantine_eval_pool": {
      "rows": 86,
      "wer": 0.6396849148368757,
      "cer": 0.29375982294797176
    }
  },
  "object_dump_predictions": 0,
  "object_dump_prediction_uids": []
}


## 4. Train LoRA Adapters


In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import train_model

training_summary = train_model(config, manifest_state)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))


2026-06-23 18:08:59,750 | INFO | qwen3_asr_levantine_run | Loading Qwen3-ASR from /home/MohammadNabulsi/.cache/huggingface/hub/models--Qwen--Qwen3-ASR-0.6B/snapshots/5eb144179a02acc5e5ba31e748d22b0cf3e303b0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
2026-06-23 18:09:02,589 | INFO | qwen3_asr_levantine_run | model=Qwen3ASRForConditionalGeneration processor=Qwen3ASRProcessor adapter=None
/home/MohammadNabulsi/whisper/Runs/qwen3_asr_0_6b_levantine_custom_streaming_5minckpt/pipeline.py:1281: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `_QwenTrainer.__init__`. Use `processing_class` instead.
  trainer = _QwenTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'pad

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

: 

## 5. Tuned Validation and Test Predictions


In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import load_qwen_bundle, run_predictions

tuned_bundle = load_qwen_bundle(config, adapter_path=training_summary["best_checkpoint"])
val_prediction_metrics = run_predictions(
    manifest_state["val_rows"],
    config,
    tuned_bundle,
    name="tuned_val_predictions",
) if config.run_post_train_eval else None
test_prediction_metrics = run_predictions(
    manifest_state["test_rows"],
    config,
    tuned_bundle,
    name="tuned_test_predictions",
) if config.run_post_train_eval else None
print("Validation metrics:")
print(json.dumps(val_prediction_metrics, ensure_ascii=False, indent=2))
print("Test metrics:")
print(json.dumps(test_prediction_metrics, ensure_ascii=False, indent=2))


## 6. Summary Report

In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import write_summary_report

summary_report = write_summary_report(
    config,
    selection_summary,
    baseline_test_metrics,
    val_prediction_metrics,
    test_prediction_metrics,
    training_summary,
)
print(json.dumps(summary_report, ensure_ascii=False, indent=2))


## 7. Integrity Report

In [ ]:
from Runs.qwen3_asr_0_6b_levantine_custom_streaming_5minckpt.pipeline import write_integrity_report

integrity_report = write_integrity_report(
    config,
    selection_summary,
    baseline_test_metrics,
    val_prediction_metrics,
    test_prediction_metrics,
    training_summary,
)
print(json.dumps(integrity_report, ensure_ascii=False, indent=2))
if integrity_report.get("prediction_health", {}).get("test_predictions", {}).get("object_dump_predictions", 0):
    raise RuntimeError("Decode guard failed: found object-dump predictions in test output.")
if integrity_report.get("prediction_health", {}).get("test_predictions", {}).get("empty_predictions", 0):
    raise RuntimeError("Integrity check failed: found empty predictions in test output.")
print("QWEN3-ASR 0.6B LEVANTINE RUN INTEGRITY CHECK PASSED")
